# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described by a Croissant schema located at the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

**Note:** All referencing of record sets, fields, and columns is done using their `@id` for clarity and reproducibility.

In [ ]:
# Install the mlcroissant library
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Let's review the available record sets, their `@id`s, fields, and columns as defined in the Croissant schema. We reference all entities by their `@id`.

In [ ]:
# List all record sets and their fields using their @id
print('Record Sets:')
for record_set in dataset.record_sets:
    print(f"  - @id: {record_set['@id']}")
    if 'field' in record_set:
        field_ids = record_set['field']
        if isinstance(field_ids, dict):
            field_ids = [field_ids]
        print("    Fields:")
        for field_ref in field_ids:
            print(f"      - @id: {field_ref['@id'] if isinstance(field_ref, dict) and '@id' in field_ref else field_ref}")
    if 'column' in record_set:
        column_ids = record_set['column']
        if isinstance(column_ids, dict):
            column_ids = [column_ids]
        print("    Columns:")
        for column_ref in column_ids:
            print(f"      - @id: {column_ref['@id'] if isinstance(column_ref, dict) and '@id' in column_ref else column_ref}")

For a quick example, let's iterate through record set(s) and display the first few records using their `@id`.

> **Note:** The record set `@id` values will be listed in the cell above. Replace `<record_set_id>` below with an actual `@id` from the output.

In [ ]:
# Example: Print the first 3 records for a record set, referencing record_set by @id
# (Replace the value below with a specific @id obtained above)
example_record_set_id = None
for record_set in dataset.record_sets:
    example_record_set_id = record_set['@id']
    break

print(f"First 3 records in record set @id='{example_record_set_id}':")
try:
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        if i >= 3:
            break
        print(record)
except Exception as e:
    print(f"Could not load records for record set {example_record_set_id}: {e}")

## 3. Data Extraction

Load data from all available record sets into individual pandas DataFrames, using their respective `@id` identifiers. This enables further analysis and processing.

In [ ]:
# Extract data into DataFrames for each record set
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for rid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rid))
        if len(records) > 0:
            dataframes[rid] = pd.DataFrame(records)
            print(f"Loaded record set @id='{rid}' with shape: {dataframes[rid].shape}")
        else:
            print(f"Record set @id='{rid}' contains no records.")
    except Exception as e:
        print(f"Could not load records for {rid}: {e}")

# If a DataFrame is loaded, print column names and preview the data (show first such DataFrame)
previewed = False
for rid, df in dataframes.items():
    print(f"\nColumns for record set @id='{rid}':\n{df.columns.tolist()}")
    print(df.head())
    previewed = True
    break
if not previewed:
    print('No record sets loaded as DataFrames for preview.')

## 4. Exploratory Data Analysis (EDA)

Perform common data processing and transformations on your DataFrame(s): filter numeric fields, normalize, and group by relevant attributes—all while referencing fields by their `@id`.

In [ ]:
# Example: 
# We'll select the first available DataFrame and perform some EDA operations. 

# Step 1: Find a record set with numeric fields
target_rid = None
numeric_field_id = None
group_field_id = None

for rid, df in dataframes.items():
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            target_rid = rid
            numeric_field_id = col
            # Attempt to find a non-numeric field for grouping as well
            non_numeric = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
            group_field_id = non_numeric[0] if non_numeric else None
            break
    if target_rid:
        break

if target_rid and numeric_field_id:
    print(f"Performing EDA on record set @id='{target_rid}' and numeric field @id='{numeric_field_id}'")
    # Example: filter records with numeric_field > threshold
    threshold = df[numeric_field_id].dropna().quantile(0.75)  # Use 75th percentile as threshold example
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field (Z-score)
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Grouping by the group field if present
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by '{group_field_id}':")
        print(grouped_df.head())
    else:
        print("\nNo suitable string field for grouping found.")
else:
    print('No suitable record set and numeric field found for EDA.')

## 5. Visualization

Visualize the distribution of a numeric field and explore potential relationships between fields. If possible, also create grouped visualizations.

> **Note:** Adjust the visualization as necessary to your data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram and boxplot for the selected numeric field
if target_rid and numeric_field_id:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, ax=axes[0])
    axes[0].set_title(f"Distribution of {numeric_field_id}")
    sns.boxplot(x=df[numeric_field_id].dropna(), ax=axes[1])
    axes[1].set_title(f"Boxplot of {numeric_field_id}")
    plt.show()

    # If we have grouping, visualize group means
    if group_field_id and group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(f"{group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.show()
else:
    print('No numeric field available to visualize.')

## 6. Conclusion

In this notebook, we've:

- Loaded dataset metadata describing regression results for rangeland management adoption predictors in Northern Kenya using `mlcroissant`.
- Examined the schema structure using `@id`-based referencing for record sets and fields.
- Extracted data into pandas DataFrames for flexible analysis.
- Demonstrated EDA and visualization techniques, including filtering by field values, normalization, and grouped summary statistics.

This simple workflow illustrates the power of Croissant schemas and `mlcroissant` for interoperable data discovery and analysis. For further insight, consider expanding the EDA or incorporating model fitting using fields of interest.

> _Remember_: For robust analyses, refer to the full data dictionary and methodology appendix in the FAIR² dataset documentation.